# 1. Histogram of Oriented Gradients (HOG) & SVM Pipeline
### Improved Object Identification and Detection

This notebook addresses and fixes the key limitations found in the original `mine/hog.ipynb`:
1. **Missing Classes**: Fixed the `.jpeg` file glob bug that previously omitted all **164 training apples** and **54 test apples**.
2. **Dimension & Model Explosion**: Replaced the huge `(512, 512)` window size (which generated **142,884 features** and a **628 MB** SVM model!) with a standardized `(128, 128)` window (8,100 features, ~39 MB model, fast training and inference).
3. **Rigorous Test Evaluation**: Added systematic test evaluation on the full test set with **Accuracy**, **Precision**, **Recall**, **F1-score**, and a **Confusion Matrix**.
4. **Object Detection**: Implemented multiscale sliding-window detection with **Non-Maximum Suppression (NMS)** to draw localized bounding boxes around detected objects.



In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

from dataset import load_split
from hog_pipeline import HOGPipeline, non_max_suppression_fast

def show_rgb(img, title=None):
    if len(img.shape) == 3:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img_rgb = img
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb, cmap='gray' if len(img.shape) == 2 else None)
    plt.axis('off')
    if title:
        plt.title(title, fontsize=12)
    plt.show()



## Step 1: Load Dataset (with .jpeg support)
Notice that the dataset contains `apples`, `dogs`, and `people`. All 3 classes are loaded dynamically.



In [ ]:
# Load train and test splits
train_imgs, train_lbls, classes, _ = load_split("train")
test_imgs, test_lbls, _, _ = load_split("test")

print(f"Classes Discovered: {classes}")
print(f"Training set: {len(train_imgs)} images")
print(f"Testing set:  {len(test_imgs)} images")

for c in classes:
    print(f"  - {c}: {train_lbls.count(c)} train, {test_lbls.count(c)} test")



## Step 2: Configure HOG Descriptor
We use:
- `win_size = (128, 128)`
- `block_size = (16, 16)`
- `block_stride = (8, 8)`
- `cell_size = (8, 8)`
- `nbins = 9`

Feature Dimension = $((128-16)/8 + 1) \times ((128-16)/8 + 1) \times 4 \times 9 = 15 \times 15 \times 36 = 8,100$.



In [ ]:
pipeline = HOGPipeline(win_size=(128, 128))
sample_des = pipeline.compute_descriptor(train_imgs[0])
print(f"HOG Descriptor dimension per image: {sample_des.shape[0]} features")



## Step 3: Train SVM Classifier
We train OpenCV's SVM (`cv2.ml.SVM`) with a Linear kernel.



In [ ]:
print("Training HOG + SVM on training set...")
train_info = pipeline.train(train_imgs, train_lbls, kernel_type="linear", C=1.0)
print(f"Training completed in {train_info['train_time_sec']:.2f} seconds!")
print(f"Feature dimension: {train_info['feature_dim']}")



## Step 4: Evaluate on Full Test Set
We evaluate accuracy, per-class precision/recall, and the confusion matrix on all 140 test images.



In [ ]:
eval_results = pipeline.evaluate(test_imgs, test_lbls)
print(f"HOG Test Accuracy: {eval_results['accuracy'] * 100:.2f}%
")
print("Classification Report:")
print(classification_report(test_lbls, eval_results['predictions'], target_names=pipeline.classes_))

# Plot Confusion Matrix
cm = np.array(eval_results["confusion_matrix"])
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(classes)))
ax.set_yticks(range(len(classes)))
ax.set_xticklabels(classes)
ax.set_yticklabels(classes)
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.title("HOG Confusion Matrix")
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="red" if i != j else "black")
plt.colorbar(im)
plt.show()



## Step 5: Save Model & Verify File Footprint
The model is saved to `hog_svm_model.yml`. Notice how compact it is compared to the original 628 MB model!



In [ ]:
pipeline.save_model("hog_svm_model.yml")
model_size_mb = Path("hog_svm_model.yml").stat().st_size / (1024 * 1024)
print(f"Saved HOG SVM Model: hog_svm_model.yml")
print(f"Model File Size:     {model_size_mb:.2f} MB")



## Step 6: Object Detection in Scenes (Sliding Window & Bounding Box)
We can scan sliding windows over a scene image to detect and localize objects.



In [ ]:
# Test sliding-window detector on an apple test image
test_scene = test_imgs[0].copy()
detections = pipeline.detect_multiscale(test_scene, min_win=(64, 64), max_win=(160, 160), step_size=16)

annotated = test_scene.copy()
for det in detections:
    x1, y1, x2, y2 = det['box']
    cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(annotated, f"{det['class']}: {det['score']:.1f}", (x1, y1 - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

show_rgb(annotated, title=f"HOG Multiscale Detections ({len(detections)} found)")

